## Table of Contents
* [1. First Look at the Data (Schemas & Previews)](#1-first-look-at-the-data-schemas-previews)
* [2. Date Ranges & Timeline Check](#2-date-ranges-timeline-check)
  * [What the date ranges tell us](#what-the-date-ranges-tell-us)
* [3. Making Sure Every Row is Unique (Primary Key Check)](#3-making-sure-every-row-is-unique-primary-key-check)
  * [Check Results](#check-results)
* [4. Checking Holidays for Duplicate Dates (Fan-Out Prevention)](#4-checking-holidays-for-duplicate-dates-fan-out-prevention)
  * [Duplicate Dates Found](#duplicate-dates-found)
  * [How we will clean the `holidays_events` table](#how-we-will-clean-the-holidays_events-table)

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
from google.cloud import bigquery

# 1. First Look at the Data (Schemas & Previews)
Connect to BigQuery, list all the tables in our dataset, and check their schemas to see what columns and data types we have.

In [2]:
client = bigquery.Client(project='store-sales-project')
dataset_ref = client.dataset('store_sales', project='store-sales-project')
dataset = client.get_dataset(dataset_ref)
table_names = [table.table_id for table in client.list_tables(dataset)]
print(*table_names, sep='\n')

holidays_events
oil
sample_submission
stores
test
train
transactions


In [3]:
for table_name in table_names:
    table_ref = dataset_ref.table(table_name)
    table = client.get_table(table_ref)
    print(f"Schema for the table '{table_name}':")
    print(*table.schema, sep='\n', end='\n\n')

Schema for the table 'holidays_events':
SchemaField('date', 'DATE', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('type', 'STRING', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('locale', 'STRING', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('locale_name', 'STRING', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('description', 'STRING', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('transferred', 'BOOLEAN', 'NULLABLE', None, None, (), None, None, None, None)

Schema for the table 'oil':
SchemaField('date', 'DATE', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('dcoilwtico', 'FLOAT', 'NULLABLE', None, None, (), None, None, None, None)

Schema for the table 'sample_submission':
SchemaField('id', 'INTEGER', 'NULLABLE', None, None, (), None, None, None, None)
SchemaField('sales', 'FLOAT', 'NULLABLE', None, None, (), None, None, None, None)

Schema for the table 'stores':
SchemaField(

In [4]:
for table_name in table_names:
    table_ref = dataset_ref.table(table_name)
    table = client.get_table(table_ref)

    name = f'{table_name}_preview'
    df_preview = client.list_rows(table, max_results=5).to_dataframe()

    print(f"\n==================== {name.upper()} ====================")
    print(f"Total Rows in Table: {table.num_rows:,} | Total Columns: {len(table.schema)}")
    display(df_preview)


==================== HOLIDAYS_EVENTS_PREVIEW ====================
Total Rows in Table: 350 | Total Columns: 6


,date,type,locale,locale_name,description,transferred
0,2012-07-23,Holiday,Local,Cayambe,Cantonizacion de Cayambe,False
1,2013-07-23,Holiday,Local,Cayambe,Cantonizacion de Cayambe,False
2,2014-07-23,Holiday,Local,Cayambe,Cantonizacion de Cayambe,False
3,2015-07-23,Holiday,Local,Cayambe,Cantonizacion de Cayambe,False
4,2016-07-23,Holiday,Local,Cayambe,Cantonizacion de Cayambe,False



==================== OIL_PREVIEW ====================
Total Rows in Table: 1,218 | Total Columns: 2


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-21,NaN
2,2013-02-18,NaN
3,2013-03-29,NaN
4,2013-05-27,NaN



==================== SAMPLE_SUBMISSION_PREVIEW ====================
Total Rows in Table: 28,512 | Total Columns: 2


,id,sales
0,3000888,0.0
1,3000889,0.0
2,3000890,0.0
3,3000891,0.0
4,3000892,0.0


... (4 outputs left out)


==================== TRAIN_PREVIEW ====================
Total Rows in Table: 3,000,888 | Total Columns: 6


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0



==================== TRANSACTIONS_PREVIEW ====================
Total Rows in Table: 83,488 | Total Columns: 3


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


# 2. Date Ranges & Timeline Check
Before joining anything, we need to know the start and end dates for each table. 

> **Note on query cost:** Since these tables are tiny (under 125 MB total), scanning the date columns here is basically free. In a real project with huge terabyte-scale tables, we'd look up these dates from table metadata or partition logs instead of running full column scans.

In [5]:
date_ranges_query = """ 
    SELECT 
        'train' AS table_name,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM `store-sales-project.store_sales.train`

    UNION ALL

    SELECT 
        'test' AS table_name,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM `store-sales-project.store_sales.test`

    UNION ALL

    SELECT 
        'holidays_events' AS table_name,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM `store-sales-project.store_sales.holidays_events`

    UNION ALL

    SELECT 
        'oil' AS table_name,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM `store-sales-project.store_sales.oil`

    UNION ALL

    SELECT 
        'transactions' AS table_name,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM `store-sales-project.store_sales.transactions`            
"""
#Set a safe limit of 100 MB
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=100*1024*1024)
query_job = client.query(date_ranges_query, job_config=safe_config)
df_date_ranges = query_job.to_dataframe()
display(df_date_ranges)

,table_name,min_date,max_date
0,train,2013-01-01,2017-08-15
1,test,2017-08-16,2017-08-31
2,holidays_events,2012-03-02,2017-12-26
3,oil,2013-01-01,2017-08-31
4,transactions,2013-01-01,2017-08-15


### What the date ranges tell us:

* **Target Forecast Window:** 
  Our `train` data goes from `2013-01-01` to `2017-08-15`, while the `test` set covers the next 16 days (`2017-08-16` to `2017-08-31`).

* **Filtering Holidays:** 
  The `holidays_events` table starts way back in 2012 and goes past our test set. For our training and testing, we only care about dates between `2013-01-01` and `2017-08-31`.

> ⚠️
> **Pipeline Action Item:** When preparing `holidays_events`, add `WHERE date BETWEEN '2013-01-01' AND '2017-08-31'`. Anything outside this window is just extra noise we don't need.

* **Why `transactions` stays out of the join:** 
  We want to forecast store sales for the 16-day window (2017-08-16 to 2017-08-31). 
  While calendar holidays and oil prices are already known for those future dates, store transactions are obviously not. 
  If we join same-day transactions during training, the model will depend on information that won't exist at test time (data leakage). To keep things safe and maintain identical schemas between train and test, we'll exclude `transactions` from our baseline join. We can always build lagged features (e.g. transactions from 16+ days ago) later on. 

* **A practical note on `oil` prices:** 
  The Kaggle test set includes oil prices for the forecast window, so we can join them directly for this competition. However, in a real-world forecasting pipeline where tomorrow's oil price is unknown, using current-day prices would fail in production. In practice, we would either use lagged prices (e.g. oil prices from 16+ days ago) or forward-fill the latest known spot price.

# 3. Making Sure Every Row is Unique (Primary Key Check)
We want to confirm that each row in `train` and `test` is uniquely identified by `(date, store_nbr, family)`, and that there are no hidden duplicate entries.

In [6]:
check_duplicates_query = """ 
    SELECT 
        'train' AS dataset,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_ids,
        (
            SELECT COUNT(*) 
            FROM (
                SELECT DISTINCT date, store_nbr, family 
                FROM `store-sales-project.store_sales.train`
            )
        ) AS unique_triplets
    FROM `store-sales-project.store_sales.train`

    UNION ALL

    SELECT 
        'test' AS dataset,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_ids,
        (
            SELECT COUNT(*) 
            FROM (
                SELECT DISTINCT date, store_nbr, family 
                FROM `store-sales-project.store_sales.test`
            )
        ) AS unique_triplets
    FROM `store-sales-project.store_sales.test`        
"""
#Set a safe limit of 120 MB
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=120*1024*1024)
query_job = client.query(check_duplicates_query, job_config=safe_config)
df_check_duplicates = query_job.to_dataframe()
display(df_check_duplicates)

,dataset,total_rows,unique_ids,unique_triplets
0,train,3000888,3000888,3000888
1,test,28512,28512,28512


<a id="section-3-1"></a>
### Check Results
The counts match 1:1 across the board:
* `train`: exactly 3,000,888 total rows, unique IDs, and unique `(date, store_nbr, family)` combinations.
* `test`: exactly 28,512 total rows, unique IDs, and unique triplets.

This proves that `(date, store_nbr, family)` is our true unique primary key with zero duplicates. The `id` column is just a row identifier for competition submissions.

# 4. Checking Holidays for Duplicate Dates (Fan-Out Prevention)
Before joining `holidays_events` with our sales tables, we need to check if any calendar date appears more than once. If a date has multiple events, a simple `LEFT JOIN` would duplicate our sales rows.

In [7]:
check_duplicate_holiday_dates = """ 
    SELECT
        date,
        COUNT(*) AS date_cnt 
    FROM `store-sales-project.store_sales.holidays_events`
    GROUP BY date    
    HAVING COUNT(*) > 1  
    ORDER BY date ASC 
"""
#Set a safe limit of 50 MB
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=50*1024*1024)
query_job = client.query(check_duplicate_holiday_dates, job_config=safe_config)
df_check_duplicate_holiday_dates = query_job.to_dataframe()
display(df_check_duplicate_holiday_dates)

,date,date_cnt
0,2012-06-25,3
1,2012-07-03,2
2,2012-12-22,2
3,2012-12-24,2
4,2012-12-31,2
5,2013-05-12,2
6,2013-06-25,3
7,2013-07-03,2
8,2013-12-22,2
9,2014-06-25,4


### Duplicate Dates Found
We found **31 dates** that have more than one event listed (some have up to 4). 
If we did a raw `LEFT JOIN` on `date`, every sales record on those 31 days would be duplicated, inflating row counts and corrupting total sales. We must deduplicate this table before joining.

In [8]:
check_holiday_types = """ 
    SELECT
        type,
        locale,
        transferred,
        COUNT(*) AS count 
    FROM `store-sales-project.store_sales.holidays_events`
    GROUP BY type, locale, transferred    
"""
#Set a safe limit of 10 MB
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10*1024*1024)
query_job = client.query(check_holiday_types, job_config=safe_config)
df_check_holiday_types = query_job.to_dataframe()
display(df_check_holiday_types)

,type,locale,transferred,count
0,Holiday,Local,False,133
1,Holiday,Local,True,4
2,Transfer,Local,False,4
3,Additional,Local,False,11
4,Holiday,National,False,52
5,Holiday,National,True,8
6,Event,National,False,56
7,Additional,National,False,40
8,Bridge,National,False,5
9,Work Day,National,False,5


### How we will clean the `holidays_events` table

> ⚠️
> **Pipeline Action Items for the Holiday Join:**
> 1. **Filter out work days:** Records where `type = 'Work Day'` or `transferred = True` are regular business days, so we should not treat them as holidays.
> 2. **Handle multi-event days:** For dates with multiple entries, we will consolidate them into a single record per calendar day so each date appears at most once.